In [ ]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_DistilBERT"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


Loading the dataset

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
mlb = joblib.load("mlb.joblib")

In [ ]:
#reusing the  aapd's mlb
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

In [ ]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [ ]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
type(tokenizer)

transformers.models.bert.tokenization_bert.BertTokenizer

In [ ]:
# for DistilBER max token length is 512 - the longest abstract has 522 words, so truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(aapd_X_train)
dev_enc   = tokenize(aapd_X_val)
test_enc  = tokenize(aapd_X_test)

In [ ]:
#for diagnostics
def get_token_length_stats(texts, name):
    lengths = np.array([
        len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
        for text in texts
    ])
    print(
        f"{name}: median={np.median(lengths):.0f}, "
        f"p95={np.percentile(lengths, 95):.0f}, "
        f"max={lengths.max()}, "
        f">512={(lengths > 512).mean():.1%}")
    return lengths

train_token_lengths = get_token_length_stats(aapd_X_train, "Train")
val_token_lengths = get_token_length_stats(aapd_X_val, "Validation")
test_token_lengths = get_token_length_stats(aapd_X_test, "Test")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (528 > 512). Running this sequence through the model will result in indexing errors


Train: median=180, p95=328, max=655, >512=0.0%
Validation: median=176, p95=326, max=541, >512=0.1%
Test: median=181, p95=326, max=465, >512=0.0%


In [ ]:
y_train_bin = aapd_y_train.astype(np.float32)
y_dev_bin   = aapd_y_val.astype(np.float32)
y_test_bin  = aapd_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)

['Adaptation and Self-Organizing Systems' 'Applications'
 'Artificial Intelligence' 'Combinatorics' 'Computation and Language'
 'Computational Complexity'
 'Computational Engineering, Finance, and Science'
 'Computational Geometry' 'Computational Linguistics'
 'Computer Science and Game Theory'
 'Computer Vision and Pattern Recognition' 'Computers and Society'
 'Cryptography and Security' 'Data Analysis, Statistics and Probability'
 'Data Structures and Algorithms' 'Databases' 'Digital Libraries'
 'Discrete Mathematics' 'Disordered Systems and Neural Networks'
 'Distributed, Parallel, and Cluster Computing'
 'Formal Languages and Automata Theory' 'Human-Computer Interaction'
 'Information Retrieval' 'Information Theory (Computer Science)'
 'Information Theory (Mathematics)' 'Logic' 'Logic in Computer Science'
 'Machine Learning (Computer Science)' 'Machine Learning (Statistics)'
 'Mathematical Software' 'Methodology' 'Multiagent Systems' 'Multimedia'
 'Networking and Internet Architect

In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
print(train_dataset[0])
print(len(train_dataset[0]["labels"]))

{'input_ids': [101, 1996, 7189, 2090, 12874, 1005, 1055, 16902, 19064, 1998, 5474, 2239, 1005, 1055, 2522, 11493, 2063, 5468, 2003, 3936, 2241, 2006, 1996, 2367, 2825, 5300, 1997, 1996, 2407, 1997, 1996, 1048, 2487, 13373, 1998, 1996, 1048, 2475, 13373, 1997, 1037, 9207, 2122, 2367, 5300, 10750, 1037, 16994, 2546, 1997, 6233, 3442, 3210, 2029, 2433, 2362, 1037, 6112, 1997, 2685, 1010, 2108, 1996, 10847, 7189, 1996, 9373, 3463, 2024, 7718, 2114, 1996, 3166, 2522, 11091, 4262, 2426, 2484, 12367, 3388, 7277, 7066, 2005, 3183, 2048, 21520, 2064, 2022, 3833, 1010, 2241, 2006, 2522, 22921, 1996, 2004, 24335, 12589, 14404, 8185, 1998, 1996, 19490, 2522, 11091, 8185, 2119, 4973, 3294, 12210, 1996, 9373, 3463, 1996, 3463, 9585, 2149, 2000, 20648, 2019, 9896, 2029, 3640, 1037, 11207, 3643, 2005, 1996, 2522, 11493, 2063, 2682, 2029, 3904, 1997, 1996, 7978, 12874, 16902, 2015, 2052, 2022, 4997, 2478, 2023, 11207, 3643, 2064, 2022, 3517, 2000, 23569, 27605, 4371, 1996, 5107, 3989, 1997, 1996, 9207,

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int) #deafult for now

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [ ]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (53840, 54)
Val labels: (1000, 54)
Test labels: (1000, 54)
Number of labels: 54


In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)


    if measure_vram:
        reset_cuda_peak_memory()


    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id)
    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])


#for resuming if something goes wrong
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
#includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "AAPD",
        "method": "full_finetuning",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        # single forward pass — gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        # derive predictions - needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                output_dir,
                f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [ ]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 10, #should be enough for such a large dataset
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3}

#small search on the most relevant hyperparameters
learning_rates = [1e-5, 2e-5, 3e-5]
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipingp already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed
search_results_df

Skipping lr=1e-05, bs=8 (already done)
Skipping lr=1e-05, bs=16 (already done)
Skipping lr=2e-05, bs=8 (already done)
Skipping lr=2e-05, bs=16 (already done)
Skipping lr=3e-05, bs=8 (already done)
Skipping lr=3e-05, bs=16 (already done)


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,AAPD,full_finetuning,0,0.00003,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.589473,10.0,69.944116,4.595245,NaN,66994998,66994998,0.589473,0.745822,74.539362
1,distilbert-base-uncased,AAPD,full_finetuning,0,0.00002,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.588487,8.0,6312.107300,4.791224,NaN,66994998,66994998,0.588487,0.753952,6316.898523
2,distilbert-base-uncased,AAPD,full_finetuning,0,0.00002,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.581087,10.0,7220.657033,4.601288,NaN,66994998,66994998,0.581087,0.747622,7225.258320
3,distilbert-base-uncased,AAPD,full_finetuning,0,0.00003,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.579331,10.0,810.742309,4.704095,NaN,66994998,66994998,0.579331,0.741513,815.446404
4,distilbert-base-uncased,AAPD,full_finetuning,0,0.00001,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.568794,10.0,7992.253319,4.798695,NaN,66994998,66994998,0.568794,0.755775,7997.052014
5,distilbert-base-uncased,AAPD,full_finetuning,0,0.00001,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.546700,10.0,5131.309829,4.370762,NaN,66994998,66994998,0.546700,0.752634,5135.680591


In [ ]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 3e-05
Best batch size: 8
Best validation macro-F1: 0.5894729427548527
Best checkpoint: /content/drive/MyDrive/thesis_results/AAPD_DistilBERT/search/lr_3e-05_bs_8/checkpoint-47110


In [ ]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [ ]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [ ]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "AAPD_DistilBERT_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"AAPD_DistilBERT_test_seed_{seed}"))

    # Skip already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    # Save incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Skipping seed=0 (already done)
Skipping seed=1 (already done)
Final run: seed=2, lr=3e-05, batch_size=8


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training from scratch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.013162,0.071682,10,0.750436,0.597934


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT/test_predictions_seed_2.npz


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_macro,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,total_measured_time_sec
0,distilbert-base-uncased,AAPD,full_finetuning,0,0.00003,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.596676,10.0,...,0.596676,0.747462,5.196214,5.196214,0.587993,0.729289,2.178,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,8016.797246
1,distilbert-base-uncased,AAPD,full_finetuning,1,0.00003,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.598423,8.0,...,0.598423,0.753458,4.981536,4.981536,0.566492,0.728468,2.142,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,6432.122444
2,distilbert-base-uncased,AAPD,full_finetuning,2,0.00003,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.597934,10.0,...,0.597934,0.750436,4.844908,4.844908,0.578110,0.731230,2.174,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,7969.350574


In [ ]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "AAPD_DistilBERT_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,total_measured_time_sec
mean,0.577532,0.729662,2.164667,2.421,7462.963123,4.786079,5.007553,7472.756754
std,0.010762,0.001418,0.019732,0.000,901.267532,0.541776,0.177092,901.527938
